# E1 · CLIP zero-shot como referencia visual

**Objetivo.** Medir la recuperación de identidades individuales con CLIP ViT-B/32 preentrenado, sin detector ni adaptación al dominio.

**Pregunta.** ¿Qué capacidad de recuperación ofrece la representación visual general de CLIP antes de introducir localización, MLP o LoRA?

**Hipótesis descriptiva.** CLIP superará el ordenamiento uniforme, pero el fondo y el recorte central estándar limitarán la identificación de animales visualmente similares.

**Intervención y controles.** E1 no entrena parámetros. Usa el modelo inmutable `clip-vit-b32@v1`, imagen completa y similitud coseno. Evalúa por separado el test público v1 y PetTrace v5; nunca mezcla sus galerías.

La tarea es recuperación *closed-set*, de la misma especie y *leave-one-photo-out*. Rank-k opera sobre identidades mediante el máximo score de sus fotos; mAP ordena fotos. La EER es descriptiva, no una calibración independiente.

### Entorno

Ejecuta el cuaderno en una sesión nueva de Colab con Python 3.11–3.13. La release exacta se instala desde PyPI; no se clona el repositorio, no se copian requirements y no se monta Drive.

In [ ]:
import sys

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("E1 requires Python 3.11–3.13")
%pip install -q --no-cache-dir "pettrace-ml[notebooks,tracking]==0.2.32"

In [ ]:
import os
import pettrace
from google.colab import userdata

EXPECTED_PACKAGE = "0.2.32"
LOG_TO_MLFLOW = True
if pettrace.__version__ != EXPECTED_PACKAGE:
    raise RuntimeError(f"Expected PetTrace {EXPECTED_PACKAGE}, found {pettrace.__version__}")

os.environ["MLFLOW_TRACKING_URI"] = "https://ml.pettrace.app"
for name in ("HF_TOKEN", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    value = userdata.get(name)
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f"Missing Colab Secret: {name}")
    os.environ[name] = value
print(f"PetTrace {pettrace.__version__} · MLflow publication required")

## Fase 1. Preparación y preprocesamiento de datos

E1 consume dos benchmarks publicados e inmutables. El test público procede de las fuentes refinadas Dogs of the World y Labeled Cats in the Wild; PetTrace v5 funciona como evaluación externa privada. El runner verifica etiquetas, commits, fingerprints y ausencia de imágenes idénticas entre dominios antes de inferir.

No se limpia, divide ni modifica información en este cuaderno. Tampoco se usa el test público para elegir parámetros: E1 no tiene entrenamiento ni selección de checkpoint.

In [ ]:
from IPython.display import Markdown, display
from pettrace.evaluation.research import (
    PUBLIC_COMMIT, PUBLIC_FINGERPRINT, PUBLIC_REPO, PUBLIC_VERSION,
)
from pettrace.experiments.e1 import PRIVATE_COMMIT, PRIVATE_REPO, PRIVATE_VERSION
from pettrace.ui import markdown_table

DATASET_COMMIT = "1777734dbffda0df0bf43305233809ee73157d94"
assert DATASET_COMMIT == PRIVATE_COMMIT
display(Markdown(markdown_table(
    ["Dominio", "Repositorio", "Versión", "Commit", "Uso"],
    [
        ["Público", PUBLIC_REPO, PUBLIC_VERSION, PUBLIC_COMMIT[:12], "test inmutable"],
        ["PetTrace", PRIVATE_REPO, PRIVATE_VERSION, PRIVATE_COMMIT[:12], "evaluación externa"],
    ],
)))
print("Public fingerprint:", PUBLIC_FINGERPRINT)

## Fase 2. Representación y entrenamiento

CLIP ViT-B/32 se usa congelado conforme a [Radford et al. (2021)](https://proceedings.mlr.press/v139/radford21a.html). El pipeline aplica orientación EXIF, RGB, redimensionamiento del lado corto, recorte central y normalización CLIP; **no localiza ni recorta la mascota con YOLO**.

No hay optimizador, épocas, semillas ni ajuste de hiperparámetros. El runner exige el artefacto registrado y su hash SHA-256, reproduce su golden set y rechaza cualquier detector. Esta condición es el control para E2–E5.

In [ ]:
from pettrace.experiments.e1 import ENCODER_SHA256, MODEL

display(Markdown(markdown_table(
    ["Componente", "Condición E1"],
    [
        ["Modelo", MODEL],
        ["Encoder SHA-256", ENCODER_SHA256],
        ["Detector", "ninguno"],
        ["Parámetros entrenados", "0"],
        ["Score", "similitud coseno"],
    ],
)))

## Fase 3. Evaluación y análisis de resultados

La siguiente celda ejecuta el flujo oficial completo: descarga y verifica ambos snapshots, valida el modelo, extrae embeddings, calcula Rank-1/Rank-2/Rank-5/mAP, genera figuras y publica **obligatoriamente** dos runs separados en MLflow. La barra tiene seis pasos normalizados.

La evidencia local se escribe antes de terminar la publicación. Si MLflow falla después de evaluar, reintenta sin recalcular con `pettrace run e1 --resume <reporte>`. Una corrida solo está completa cuando `publication.json` contiene `published: true`.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from pettrace.experiments.e1 import run

if not LOG_TO_MLFLOW:
    raise RuntimeError("MLflow publication is mandatory for E1")
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
REPORT = Path("reports") / f"e1-{run_id}"
publication = run(REPORT)
if not publication.get("published"):
    raise RuntimeError("E1 is incomplete: MLflow publication did not finish")

In [ ]:
import json
from IPython.display import Image, Markdown, display

display(Markdown((REPORT / "benchmark-comparison.md").read_text()))
display(Image(filename=str(REPORT / "benchmark-comparison.png")))

for label, folder in (("Test público v1", REPORT / "public_test"), ("PetTrace v5", REPORT)):
    summary = json.loads((folder / "summary.json").read_text())
    global_metrics = summary["global"]
    display(Markdown(
        f"### {label}\n"
        f"**Rank-1:** {global_metrics['rank1']:.2%} · "
        f"**Rank-2:** {global_metrics['rank2']:.2%} · "
        f"**Rank-5:** {global_metrics['rank5']:.2%} · "
        f"**mAP:** {global_metrics['mAP']:.4f}"
    ))
    for figure in ("ranking-metrics.png", "cmc.png", "confusion.png", "score-distribution.png"):
        display(Image(filename=str(folder / figure)))

display(Markdown(
    f"MLflow: [run PetTrace]({publication['url']}) · "
    f"run público `{publication['public_run_id']}`"
))

### Discusión y limitaciones

E1 mide transferencia visual, no entrenamiento de Re-ID. Una imagen completa mezcla identidad, fondo, pose e iluminación; la evaluación no separa causalmente esos factores ni garantiza independencia entre sesiones de captura. PetTrace y el test público tienen identidades y galerías distintas, por lo que su diferencia es descriptiva.

La comparación válida con experimentos posteriores exige conservar exactamente estos snapshots y este protocolo. E2 cambia únicamente la localización; E3 añade una cabeza MLP; E4 adapta CLIP con LoRA.

**Evidencia registrada con PetTrace 0.2.31 (2026-11-07).** El test público obtuvo Rank-1 68,59 %, Rank-2 80,14 % y mAP 0,5501 ([MLflow](https://ml.pettrace.app/#/experiments/3/runs/7ec775569d79464da22413d804b1d51a)); PetTrace v5 obtuvo Rank-1 65,41 %, Rank-2 81,65 % y mAP 0,4067 ([MLflow](https://ml.pettrace.app/#/experiments/3/runs/f650a1f85663448fa4783d7678a8670c)). Estas cifras son evidencia de esa corrida publicada, no salidas de una sesión nueva del cuaderno. Las salidas ejecutadas de la presentación histórica v4 permanecen [inmutables en el commit anterior](https://github.com/alessandro54/pettrace-ml/blob/c7c265ae26a802ebcdc5ada41d79bd7cbe238271/notebooks/e1_clip_zero_shot.ipynb).